# Implement Temperature Sampling

**Difficulty**: 🟢 Easy

### Problem Statement

Temperature sampling is the simplest and most fundamental decoding control for language models. By dividing the logits by a temperature parameter before applying softmax, you control the "sharpness" of the probability distribution:

- **Low temperature** (e.g., 0.1): The distribution becomes peaky, concentrating probability on the most likely tokens. Approaches argmax/greedy decoding as temperature approaches 0.
- **Temperature = 1.0**: The original distribution, unchanged.
- **High temperature** (e.g., 5.0): The distribution becomes flatter/more uniform, increasing randomness and diversity.

Your goal is to implement temperature-scaled sampling and demonstrate how temperature affects the output distribution.

---

### Requirements

1. **Temperature Scaling**
   - Divide raw logits by the temperature value.
   - Handle edge case: temperature should be > 0.

2. **Softmax and Sampling**
   - Apply softmax to the scaled logits to get a valid probability distribution.
   - Sample a token index from this distribution using `torch.multinomial`.

3. **Behavioral Verification**
   - At very low temperature, sampling should almost always return the argmax token.
   - At high temperature, the distribution should be nearly uniform (high entropy).

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ The output should be a single token index (integer).
- ✅ Temperature must be positive (> 0).
- ✅ Demonstrate the effect of temperature on distribution entropy.

---

**Companies**: OpenAI, Anthropic, Cohere, Perplexity

---

<details>
  <summary>💡 Hint</summary>

  - The core operation is just `softmax(logits / temperature)`.
  - Entropy of a distribution p is `-sum(p * log(p))`. Higher entropy means more uniform.
  - At temperature close to 0, use a small epsilon (e.g., 1e-8) to avoid division by zero.
  - `torch.multinomial(probs, 1)` samples one token index from the probability distribution.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# Data generation / setup
torch.manual_seed(42)
vocab_size = 50257

# Create fake logits as if from an LLM's final layer
logits = torch.randn(1, vocab_size)
print(f"Logits shape: {logits.shape}")
print(f"Logits range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")
print(f"Argmax token: {logits.argmax(dim=-1).item()}")

In [ ]:
def temperature_sample(logits, temperature=1.0):
    """
    Perform temperature-scaled sampling on logits.
    
    Args:
        logits (Tensor): Raw logits of shape (1, vocab_size)
        temperature (float): Temperature for scaling (must be > 0)
        
    Returns:
        int: Sampled token index
    """
    # Step 1: Validate temperature
    assert temperature > 0, f"Temperature must be positive, got {temperature}"
    
    # Step 2: Apply temperature scaling
    scaled_logits = logits / temperature
    
    # Step 3: Apply softmax to get probabilities
    probs = F.softmax(scaled_logits, dim=-1)
    
    # Step 4: Sample from the distribution
    token = torch.multinomial(probs, num_samples=1)
    
    return token.item()

In [ ]:
# Validation
print("=== Temperature Sampling Validation ===")
print()

# Test 1: Output should be a valid token index
token = temperature_sample(logits, temperature=1.0)
assert isinstance(token, int) or (isinstance(token, torch.Tensor) and token.ndim == 0), \
    f"Output should be a scalar token index, got {type(token)}"
assert 0 <= token < vocab_size, f"Token {token} out of range [0, {vocab_size})"
print(f"Test 1 PASSED: Sampled token {token} is a valid index")

# Test 2: Very low temperature should approach argmax
argmax_token = logits.argmax(dim=-1).item()
num_samples = 100
low_temp_samples = [temperature_sample(logits, temperature=0.01) for _ in range(num_samples)]
low_temp_samples = [s if isinstance(s, int) else s.item() for s in low_temp_samples]
argmax_count = sum(1 for s in low_temp_samples if s == argmax_token)
print(f"Test 2: At temperature=0.01, argmax token ({argmax_token}) sampled {argmax_count}/{num_samples} times")
assert argmax_count >= 95, f"Low temperature should almost always return argmax, got {argmax_count}/100"
print(f"Test 2 PASSED: Low temperature approaches argmax")

# Test 3: High temperature should produce high entropy (near-uniform distribution)
def compute_entropy(logits, temperature):
    probs = F.softmax(logits / temperature, dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -torch.sum(probs * log_probs, dim=-1)
    return entropy.item()

max_entropy = torch.log(torch.tensor(float(vocab_size))).item()  # entropy of uniform distribution
print(f"\nTest 3: Entropy analysis (max possible = {max_entropy:.4f})")
for temp in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    entropy = compute_entropy(logits, temp)
    print(f"  temperature={temp:>5.1f} -> entropy={entropy:.4f} ({entropy/max_entropy*100:.1f}% of max)")

low_entropy = compute_entropy(logits, 0.1)
high_entropy = compute_entropy(logits, 10.0)
assert high_entropy > low_entropy, "High temperature should have higher entropy than low temperature"
print(f"Test 3 PASSED: Higher temperature produces higher entropy")

# Test 4: Temperature=1.0 should give the original distribution
probs_t1 = F.softmax(logits / 1.0, dim=-1)
probs_orig = F.softmax(logits, dim=-1)
assert torch.allclose(probs_t1, probs_orig), "Temperature=1.0 should not change the distribution"
print(f"Test 4 PASSED: Temperature=1.0 preserves original distribution")

print("\nAll tests passed!")